In [5]:
import sys
import os
import psycopg2
from dotenv import load_dotenv
import QuantLib as ql
import pandas as pd

sys.path.append(os.path.abspath(".."))
from utils.black_scholes import get_implied_vol

load_dotenv()

url = os.getenv("DATABASE_URL")

In [11]:
conn = psycopg2.connect(url, sslmode="require")

query = """
SELECT ticker, timestamp, bids, asks
FROM orderbooks
"""
df = pd.read_sql(query, conn)
df.head()

C:\Users\bakae\AppData\Local\Temp\ipykernel_20360\243514560.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,ticker,timestamp,bids,asks
0,SR300CE6,2026-04-16 06:58:14+00:00,"[{'price': 25.24, 'quantity': 800}, {'price': ...","[{'price': 28.44, 'quantity': 5500}, {'price':..."
1,SR360CQ6,2026-04-16 06:58:14+00:00,"[{'price': 32.33, 'quantity': 800}]","[{'price': 34.94, 'quantity': 800}, {'price': ..."
2,SR330CP6D,2026-04-16 06:58:14+00:00,"[{'price': 7.38, 'quantity': 2500}, {'price': ...","[{'price': 9.82, 'quantity': 800}]"
3,SR320CD6E,2026-04-16 06:58:14+00:00,"[{'price': 5.9, 'quantity': 100}, {'price': 5....","[{'price': 6.37, 'quantity': 5500}, {'price': ..."
4,SR310CE6,2026-04-16 06:58:15+00:00,"[{'price': 16.51, 'quantity': 800}, {'price': ...","[{'price': 20.12, 'quantity': 5500}, {'price':..."


In [20]:
df.set_index("timestamp", inplace=True)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

In [23]:
import numpy as np 

df["best_bid"] = df["bids"].str[0].str["price"]
df["best_ask"] = df["asks"].str[0].str["price"]

df["mid"] = np.where(
    df["best_bid"].notna() & df["best_ask"].notna(),
    (df["best_bid"] + df["best_ask"]) / 2,
    np.where(
        df["best_bid"].notna(),
        df["best_bid"],
        df["best_ask"]
    )
)
df.head()

,ticker,bids,asks,mid,best_bid,best_ask
timestamp,,,,,,
2026-03-23 15:39:07+00:00,SR360CC6D,[],"[{'price': 0.54, 'quantity': 1}]",0.54,NaN,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,[],"[{'price': 0.54, 'quantity': 1}]",0.54,NaN,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,[],"[{'price': 0.54, 'quantity': 1}]",0.54,NaN,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,[],"[{'price': 0.54, 'quantity': 1}]",0.54,NaN,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,[],"[{'price': 0.54, 'quantity': 1}]",0.54,NaN,0.54


In [27]:
df = df[["ticker", "mid"]]
df.index = pd.to_datetime(df.index)
df.head()

,ticker,mid
timestamp,,
2026-03-23 15:39:07+00:00,SR360CC6D,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,0.54
2026-03-23 15:39:07+00:00,SR360CC6D,0.54


In [31]:
def get_expiry_for_ticker(df, ticker):
    dft = df[df["ticker"] == ticker].sort_index()
    
    last_row = dft.iloc[-1]
    last_timestamp = dft.index[-1]
    
    weekday = last_timestamp.weekday()
    
    if weekday == 2:  
        return last_timestamp.date()
    
    return None
    
tickers = df["ticker"].unique()

expiry_map = {}

for ticker in tickers:
    expiry_map[ticker] = get_expiry_for_ticker(df, ticker)

In [33]:
df["expiry"] = df["ticker"].map(expiry_map)
df.head()

,ticker,mid,expiry
timestamp,,,
2026-03-23 15:39:07+00:00,SR360CC6D,0.54,2026-03-25
2026-03-23 15:39:07+00:00,SR360CC6D,0.54,2026-03-25
2026-03-23 15:39:07+00:00,SR360CC6D,0.54,2026-03-25
2026-03-23 15:39:07+00:00,SR360CC6D,0.54,2026-03-25
2026-03-23 15:39:07+00:00,SR360CC6D,0.54,2026-03-25


In [40]:
df["time_bucket"] = df.index.floor("1min")
df_bucketed = (df.sort_index().groupby(["time_bucket", "ticker"]).last().reset_index())

df_bucketed.head()

,time_bucket,ticker,mid,expiry
0,2026-03-23 15:39:00+00:00,SR360CC6D,0.540,2026-03-25
1,2026-03-23 15:49:00+00:00,SR270CP6,0.780,2026-04-15
2,2026-03-23 15:49:00+00:00,SR280CP6,0.795,2026-04-15
3,2026-03-23 15:49:00+00:00,SR290CD6,31.175,2026-04-15
4,2026-03-23 15:49:00+00:00,SR290CD6A,29.495,2026-04-01


In [42]:
df_bucketed["strike"] = df_bucketed["ticker"].str[2:5]
df_bucketed.head()

,time_bucket,ticker,mid,expiry,strike
0,2026-03-23 15:39:00+00:00,SR360CC6D,0.540,2026-03-25,360
1,2026-03-23 15:49:00+00:00,SR270CP6,0.780,2026-04-15,270
2,2026-03-23 15:49:00+00:00,SR280CP6,0.795,2026-04-15,280
3,2026-03-23 15:49:00+00:00,SR290CD6,31.175,2026-04-15,290
4,2026-03-23 15:49:00+00:00,SR290CD6A,29.495,2026-04-01,290
